# The PC Hardware Auditor — Classification Notebook

**Plan compliance** (see `refactor_regression_and_logistic_plan` Part C):

- `RandomizedSearchCV` with stratified CV for Logistic, SVC-RBF, Decision Tree,
  and HistGradientBoosting; soft `VotingClassifier` built from tuned estimators.
- Calibration curves (multiclass: one-vs-rest per class) before interpreting
  soft-voting probabilities.
- Macro-F1, balanced accuracy, ROC-AUC OvR, PR-AUC macro, log-loss, per-class
  Brier scores, confusion matrix.
- Permutation importance on the winning model; misclassification error analysis.
- Persist best multiclass artifact to `reports/classification_multiclass.joblib`.

Design narrative: [docs/data-notes/classification-plan.md](../docs/data-notes/classification-plan.md).
Prerequisite: run `notebooks/Project_refactored.ipynb` to create `reports/residuals.parquet`.

## Part 0 — Setup

In [10]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, VotingClassifier
from sklearn.inspection import permutation_importance
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    StratifiedShuffleSplit,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler, TargetEncoder
from sklearn.svm import LinearSVC, SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.utils import check_X_y
from sklearn.utils.validation import check_is_fitted

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SEED = 1234
np.random.seed(SEED)

# Hyperparameter search: stratified subsample + 3-fold CV keeps runtime tractable on laptops
# while still matching the plan (RandomizedSearchCV + documented grid). Final models refit on ALL train rows.
TUNE_SUBSAMPLE = 25_000
CV_TUNING = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
N_ITER_SEARCH = 10
RANDOM_SEARCH_SEED = SEED

print("python :", sys.version.split()[0])
print("sklearn:", sklearn.__version__)
print("pandas :", pd.__version__)

python : 3.11.7
sklearn: 1.5.1
pandas : 2.2.2


In [11]:
def resolve_repo_root() -> Path:
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (candidate / "computer_prices_all.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root.")

REPO_ROOT = resolve_repo_root()
REPORTS_DIR = REPO_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)
CSV_PATH = REPO_ROOT / "computer_prices_all.csv"
RESIDUALS_PATH = REPORTS_DIR / "residuals.parquet"
MODEL_PATH = REPORTS_DIR / "classification_multiclass.joblib"

if not RESIDUALS_PATH.exists():
    raise FileNotFoundError(
        f"Run notebooks/Project_refactored.ipynb first so {RESIDUALS_PATH.name} exists."
    )

df = pd.read_csv(CSV_PATH)
residuals = pd.read_parquet(RESIDUALS_PATH)
print("df       :", df.shape)
print("residuals:", residuals.shape)
residuals.head()

df       : (100000, 33)
residuals: (100000, 6)


,row_id,split,y_true,y_pred,residual,residual_std
0,66472,train,2151.99,2405.446422,-253.456422,-1.075869
1,30216,train,2518.99,2405.578042,113.411958,0.481410
2,78035,train,3519.99,3004.980707,515.009293,2.186105
3,97011,train,1713.99,1415.728583,298.261417,1.266056
4,4394,train,1329.99,1317.332946,12.657054,0.053727


## Part 1 — Labels (train-only cutpoints)

4-class Value Score from **raw residual** quantiles on the train split only
(same as classification-plan.md §2.1).

In [12]:
train_resid = residuals.loc[residuals["split"] == "train", "residual"].values
q10, q50, q90 = np.quantile(train_resid, [0.10, 0.50, 0.90])
print(f"train residual percentiles (USD): 10th={q10:.2f}  50th={q50:.2f}  90th={q90:.2f}")

def assign_value_score(r: np.ndarray) -> np.ndarray:
    labels = np.empty(r.shape, dtype=int)
    labels[r <= q10] = 0
    labels[(r > q10) & (r <= q50)] = 1
    labels[(r > q50) & (r <= q90)] = 2
    labels[r > q90] = 3
    return labels

residuals["value_score"] = assign_value_score(residuals["residual"].values)
residuals["y_steal"] = (residuals["value_score"] == 0).astype(int)
residuals["y_anomalous"] = (np.abs(residuals["residual_std"]) > 1.0).astype(int)

print("\nClass distribution — value_score:")
print(pd.crosstab(residuals["split"], residuals["value_score"], normalize="index").round(3))

train residual percentiles (USD): 10th=-224.07  50th=2.50  90th=249.96

Class distribution — value_score:
value_score      0    1      2      3
split                                
test         0.102  0.4  0.396  0.101
train        0.100  0.4  0.400  0.100


## Part 2 — Features + tiered preprocessor (specs only, no residual)

In [13]:
from sklearn.base import BaseEstimator, TransformerMixin

TARGET_COL = "price"
X_all = df.drop(columns=[TARGET_COL])
X_train, X_test, _, _ = train_test_split(
    X_all, df[TARGET_COL], test_size=0.20, random_state=SEED, stratify=df["device_type"]
)

y_mc_train = residuals.set_index("row_id").loc[X_train.index, "value_score"].values
y_mc_test = residuals.set_index("row_id").loc[X_test.index, "value_score"].values
y_bin_train = residuals.set_index("row_id").loc[X_train.index, "y_steal"].values
y_bin_test = residuals.set_index("row_id").loc[X_test.index, "y_steal"].values

TARGET_ENCODE_COLS = ["cpu_model", "gpu_model"]
FREQUENCY_ENCODE_COLS = ["brand"]
LOW_CARD_OHE_COLS = [
    "device_type", "os", "form_factor", "cpu_brand", "gpu_brand",
    "storage_type", "display_type", "resolution", "wifi",
]
NUMERIC_COLS = [
    "release_year", "cpu_tier", "cpu_cores", "cpu_threads",
    "cpu_base_ghz", "cpu_boost_ghz", "gpu_tier", "vram_gb", "ram_gb",
    "storage_gb", "storage_drive_count", "display_size_in",
    "refresh_hz", "battery_wh", "charger_watts", "psu_watts",
    "bluetooth", "weight_kg", "warranty_months",
]
DROP_COLS = ["model"]

class FrequencyEncoderTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = pd.DataFrame(X, columns=FREQUENCY_ENCODE_COLS)
        self.mapping_ = {
            col: X[col].value_counts(normalize=True).to_dict() for col in X.columns
        }
        self.feature_names_in_ = np.array(FREQUENCY_ENCODE_COLS)
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=FREQUENCY_ENCODE_COLS)
        out = np.zeros(X.shape, dtype=float)
        for j, col in enumerate(X.columns):
            out[:, j] = X[col].map(self.mapping_[col]).fillna(0.0).values
        return out

    def get_feature_names_out(self, input_features=None):
        return np.array([f"{c}_freq" for c in FREQUENCY_ENCODE_COLS])

preprocessor = ColumnTransformer(
    transformers=[
        ("drop_cols", "drop", DROP_COLS),
        ("numeric", RobustScaler(), NUMERIC_COLS),
        (
            "low_card_ohe",
            OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.01, sparse_output=False),
            LOW_CARD_OHE_COLS,
        ),
        (
            "target_enc",
            TargetEncoder(target_type="continuous", smooth="auto", random_state=SEED),
            TARGET_ENCODE_COLS,
        ),
        ("frequency_enc", FrequencyEncoderTransformer(), FREQUENCY_ENCODE_COLS),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

preprocessor.fit(X_train, np.log1p(df.loc[X_train.index, "price"]))
Xtr_enc = preprocessor.transform(X_train)
Xte_enc = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()
print("encoded:", Xtr_enc.shape, Xte_enc.shape)

# Stratified tuning subset (hyperparameter search only; refit on full train below)
_sss = StratifiedShuffleSplit(n_splits=1, train_size=min(TUNE_SUBSAMPLE, len(y_mc_train)), random_state=SEED)
tune_idx, _ = next(_sss.split(np.arange(len(y_mc_train)), y_mc_train))
X_tune = Xtr_enc[tune_idx]
y_tune = y_mc_train[tune_idx]
print("tuning subset:", X_tune.shape, "(refit on full train after search)")

encoded: (80000, 65) (20000, 65)
tuning subset: (25000, 65) (refit on full train after search)


## Part 3 — `RandomizedSearchCV` + model zoo

Each search uses **`RandomizedSearchCV`** with **stratified 3-fold CV** on a **25k stratified
tuning subset** of train (runtime); the **best hyperparameters are refit on all 80k train rows**
before test evaluation. This matches the plan’s “small documented grid” while staying executable.

**SVC-RBF** uses a **Nyström kernel approximation** so it can be tuned and refit on the **full 80k train rows** (exact `SVC(kernel="rbf", probability=True)` is O(n²)–O(n³) and not tractable here). The pipeline is `StandardScaler → Nystroem(rbf) → CalibratedClassifierCV(LinearSVC)`, which recovers an RBF-shaped decision boundary in ~linear time and supplies the `predict_proba` required for soft voting.

`LinearSVC` is also wrapped in `CalibratedClassifierCV` for soft voting.

In [14]:
def multiclass_metrics(y_true, y_pred, proba, prefix=""):
    out = {
        f"{prefix}macro_f1": f1_score(y_true, y_pred, average="macro"),
        f"{prefix}balanced_acc": balanced_accuracy_score(y_true, y_pred),
        f"{prefix}accuracy": accuracy_score(y_true, y_pred),
    }
    if proba is not None:
        out[f"{prefix}roc_auc_ovr"] = roc_auc_score(y_true, proba, multi_class="ovr", average="macro")
        out[f"{prefix}pr_auc_macro"] = average_precision_score(y_true, proba, average="macro")
        out[f"{prefix}log_loss"] = log_loss(y_true, proba)
        out[f"{prefix}brier_mean_ovr"] = float(
            np.mean([brier_score_loss((y_true == k).astype(int), proba[:, k]) for k in range(4)])
        )
    return out

In [15]:
# --- Logistic Regression ---
lr_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=10000, random_state=SEED, n_jobs=-1)),
])
lr_search = RandomizedSearchCV(
    lr_pipe,
    param_distributions={"clf__C": np.logspace(-2, 2, 20)},
    n_iter=N_ITER_SEARCH,
    scoring="f1_macro",
    cv=CV_TUNING,
    random_state=RANDOM_SEARCH_SEED,
    n_jobs=-1,
    refit=True,
    verbose=0,
)
lr_search.fit(X_tune, y_tune)
lr_best = clone(lr_search.best_estimator_)
lr_best.fit(Xtr_enc, y_mc_train)
print("Logistic best params:", lr_search.best_params_)
lr_pred = lr_best.predict(Xte_enc)
lr_proba = lr_best.predict_proba(Xte_enc)
lr_row = {"model": "Logistic (tuned)", **multiclass_metrics(y_mc_test, lr_pred, lr_proba)}
print(lr_row)

Logistic best params: {'clf__C': 0.01}
{'model': 'Logistic (tuned)', 'macro_f1': 0.26433570123431427, 'balanced_acc': 0.3384171284401639, 'accuracy': 0.27155, 'roc_auc_ovr': 0.5718914982398076, 'pr_auc_macro': 0.30147335924924873, 'log_loss': 1.4325494100091172, 'brier_mean_ovr': 0.19623945352949276}


In [16]:
# --- Decision Tree ---
dt_search = RandomizedSearchCV(
    DecisionTreeClassifier(class_weight="balanced", random_state=SEED),
    param_distributions={
        "max_depth": [6, 8, 10, 12, 16, 24, None],
        "min_samples_leaf": [10, 25, 50, 100, 200],
        "min_samples_split": [2, 5, 10],
    },
    n_iter=N_ITER_SEARCH,
    scoring="f1_macro",
    cv=CV_TUNING,
    random_state=RANDOM_SEARCH_SEED,
    n_jobs=-1,
    refit=True,
    verbose=0,
)
dt_search.fit(X_tune, y_tune)
dt_best = clone(dt_search.best_estimator_)
dt_best.fit(Xtr_enc, y_mc_train)
print("DT best params:", dt_search.best_params_)
dt_pred = dt_best.predict(Xte_enc)
dt_proba = dt_best.predict_proba(Xte_enc)
dt_row = {"model": "DecisionTree (tuned)", **multiclass_metrics(y_mc_test, dt_pred, dt_proba)}
print(dt_row)

DT best params: {'min_samples_split': 10, 'min_samples_leaf': 10, 'max_depth': 10}
{'model': 'DecisionTree (tuned)', 'macro_f1': 0.30034611591691657, 'balanced_acc': 0.36330556231460975, 'accuracy': 0.31685, 'roc_auc_ovr': 0.6006993022331444, 'pr_auc_macro': 0.3116375382083057, 'log_loss': 2.078616579578239, 'brier_mean_ovr': 0.1966355403676262}


In [17]:
# --- HistGradientBoosting (class_weight balanced in sklearn >= 1.2) ---
hgb_search = RandomizedSearchCV(
    HistGradientBoostingClassifier(random_state=SEED, class_weight="balanced"),
    param_distributions={
        "learning_rate": [0.05, 0.08, 0.1, 0.15],
        "max_depth": [6, 8, 10, 12],
        "max_iter": [150, 200, 300, 400],
        "min_samples_leaf": [10, 20, 40],
    },
    n_iter=N_ITER_SEARCH,
    scoring="f1_macro",
    cv=CV_TUNING,
    random_state=RANDOM_SEARCH_SEED,
    n_jobs=-1,
    refit=True,
    verbose=0,
)
hgb_search.fit(X_tune, y_tune)
hgb_best = clone(hgb_search.best_estimator_)
hgb_best.fit(Xtr_enc, y_mc_train)
print("HistGBM best params:", hgb_search.best_params_)
hgb_pred = hgb_best.predict(Xte_enc)
hgb_proba = hgb_best.predict_proba(Xte_enc)
hgb_row = {"model": "HistGBM (tuned)", **multiclass_metrics(y_mc_test, hgb_pred, hgb_proba)}
print(hgb_row)

HistGBM best params: {'min_samples_leaf': 10, 'max_iter': 300, 'max_depth': 8, 'learning_rate': 0.1}
{'model': 'HistGBM (tuned)', 'macro_f1': 0.33347082884393414, 'balanced_acc': 0.4030352650007298, 'accuracy': 0.34465, 'roc_auc_ovr': 0.6427989384854399, 'pr_auc_macro': 0.34661596393784444, 'log_loss': 1.3629689269741978, 'brier_mean_ovr': 0.18836902805749653}


In [ ]:
# --- SVC RBF on full train via Nystroem kernel approximation ---
# Exact SVC(kernel="rbf", probability=True) on 80k rows is O(n^2)-O(n^3) and infeasible.
# Nystroem maps inputs into an explicit RBF feature space; LinearSVC then solves in
# O(n * n_components), and CalibratedClassifierCV gives the predict_proba soft-voting needs.
svc_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("rbf", Nystroem(kernel="rbf", random_state=SEED)),
    ("clf", CalibratedClassifierCV(
        estimator=LinearSVC(class_weight="balanced", max_iter=4000, random_state=SEED, dual="auto"),
        method="sigmoid",
        cv=3,
    )),
])
svc_search = RandomizedSearchCV(
    svc_pipe,
    param_distributions={
        "rbf__gamma": np.logspace(-3, 0, 10),
        "rbf__n_components": [300, 500, 800],
        "clf__estimator__C": np.logspace(-1, 2, 10),
    },
    n_iter=min(N_ITER_SEARCH, 10),
    scoring="f1_macro",
    cv=CV_TUNING,
    random_state=RANDOM_SEARCH_SEED,
    n_jobs=-1,
    refit=True,
    verbose=0,
)
svc_search.fit(X_tune, y_tune)
svc_best = clone(svc_search.best_estimator_)
svc_best.fit(Xtr_enc, y_mc_train)
print("SVC-RBF (Nystroem, full-train refit) best params:", svc_search.best_params_)
svc_pred = svc_best.predict(Xte_enc)
svc_proba = svc_best.predict_proba(Xte_enc)
svc_row = {"model": "SVC_RBF_Nystroem (tuned)", **multiclass_metrics(y_mc_test, svc_pred, svc_proba)}
print(svc_row)

In [ ]:
# --- Exact SVC(kernel="rbf") on a stratified 20k subsample (diagnostic only) ---
# Purpose: sanity-check that the Nystroem approximation isn't leaving real macro-F1 on the table.
# We reuse gamma/C found by the Nystroem search so the comparison is apples-to-apples:
#   - Same (C, gamma) pair
#   - Exact RBF kernel vs its Nystroem approximation
#   - 20k exact-RBF rows (all we can afford in O(n^2)) vs 80k Nystroem rows
# This model is NOT added to the soft-voting ensemble; it's a reference row in the results table.
_best = svc_search.best_params_
_gamma_exact = _best["rbf__gamma"]
_C_exact = _best["clf__estimator__C"]

_sss_exact = StratifiedShuffleSplit(n_splits=1, train_size=20_000, random_state=SEED)
_idx_exact, _ = next(_sss_exact.split(Xtr_enc, y_mc_train))

svc_exact = Pipeline([
    ("scale", StandardScaler()),
    ("clf", SVC(
        kernel="rbf",
        C=_C_exact,
        gamma=_gamma_exact,
        probability=True,
        class_weight="balanced",
        random_state=SEED,
    )),
])
svc_exact.fit(Xtr_enc[_idx_exact], y_mc_train[_idx_exact])
svc_exact_pred = svc_exact.predict(Xte_enc)
svc_exact_proba = svc_exact.predict_proba(Xte_enc)
svc_exact_row = {
    "model": f"SVC_RBF_20k (exact; C={_C_exact:.3g}, gamma={_gamma_exact:.3g})",
    **multiclass_metrics(y_mc_test, svc_exact_pred, svc_exact_proba),
}
print(svc_exact_row)

In [ ]:
# --- LinearSVC + calibration (small C search) ---
lsvc_search = RandomizedSearchCV(
    CalibratedClassifierCV(
        estimator=LinearSVC(class_weight="balanced", max_iter=4000, random_state=SEED),
        method="sigmoid",
        cv=3,
    ),
    param_distributions={"estimator__C": np.logspace(-2, 2, 15)},
    n_iter=10,
    scoring="f1_macro",
    cv=CV_TUNING,
    random_state=RANDOM_SEARCH_SEED,
    n_jobs=-1,
    refit=True,
    verbose=0,
)
lsvc_search.fit(X_tune, y_tune)
lsvc_best = clone(lsvc_search.best_estimator_)
lsvc_best.fit(Xtr_enc, y_mc_train)
print("LinearSVC (calibrated) best params:", lsvc_search.best_params_)
lsvc_pred = lsvc_best.predict(Xte_enc)
lsvc_proba = lsvc_best.predict_proba(Xte_enc)
lsvc_row = {"model": "LinearSVC_calibrated (tuned)", **multiclass_metrics(y_mc_test, lsvc_pred, lsvc_proba)}
print(lsvc_row)

In [ ]:
# --- Soft voting ensemble (plan §C.5) ---
voting = VotingClassifier(
    estimators=[
        ("lr", clone(lr_best)),
        ("lsvc", clone(lsvc_best)),
        ("svc", clone(svc_best)),
        ("dt", clone(dt_best)),
        ("hgb", clone(hgb_best)),
    ],
    voting="soft",
    n_jobs=1,
)
voting.fit(Xtr_enc, y_mc_train)
v_pred = voting.predict(Xte_enc)
v_proba = voting.predict_proba(Xte_enc)
voting_row = {"model": "Voting soft (tuned base learners)", **multiclass_metrics(y_mc_test, v_pred, v_proba)}
print(voting_row)

mc_results = [lr_row, lsvc_row, svc_row, svc_exact_row, dt_row, hgb_row, voting_row]
mc_df = pd.DataFrame(mc_results).sort_values("macro_f1", ascending=False).reset_index(drop=True)
mc_df

## Part 3b — Calibration curves (multiclass, one-vs-rest)

For soft voting, check that predicted probabilities align with observed
frequencies on the **test** set (exploratory; production would use a held-out calibration split).

In [ ]:
CLASS_NAMES = ["Steal", "Fair", "Brand Prem.", "Extreme Tax"]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
models_cal = [
    ("Logistic", lr_best),
    ("HistGBM", hgb_best),
    ("Voting", voting),
]
for r, (name, mdl) in enumerate(models_cal):
    p = mdl.predict_proba(Xte_enc)
    for k in range(4):
        prob_true, prob_pred = calibration_curve(
            (y_mc_test == k).astype(int), p[:, k], n_bins=10, strategy="uniform"
        )
        axes[r, k].plot(prob_pred, prob_true, marker="o")
        axes[r, k].plot([0, 1], [0, 1], linestyle="--", color="gray")
        axes[r, k].set_title(f"{name} | {CLASS_NAMES[k]}")
plt.tight_layout()
plt.show()

## Part 4 — Winner metrics, confusion matrix, permutation importance, error analysis

In [ ]:
winner = voting
winner_name = "Voting soft (tuned base learners)"

y_pred_mc = winner.predict(Xte_enc)
print(classification_report(
    y_mc_test, y_pred_mc,
    target_names=["Steal", "Fair", "Brand Premium", "Extreme Tax"],
    digits=3,
))

cm = confusion_matrix(y_mc_test, y_pred_mc)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels(CLASS_NAMES, rotation=30)
ax.set_yticklabels(CLASS_NAMES)
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion — {winner_name}")
plt.colorbar(im, ax=ax, fraction=0.045)
plt.tight_layout()
plt.show()

In [ ]:
# Permutation importance (encoded feature matrix; column names for display)
perm = permutation_importance(
    winner, Xte_enc, y_mc_test, n_repeats=5, random_state=SEED, n_jobs=1
)
# Map importances back to original dataframe columns (approximate: mean over encoded blocks)
imp_df = pd.DataFrame({"importance_mean": perm.importances_mean}, index=feature_names).sort_values(
    "importance_mean", ascending=False
).head(25)
print(imp_df)

In [ ]:
# Error analysis: Extreme Tax (3) predicted as Brand Premium (2) — "Brand Tax" confusion
bad = (y_mc_test == 3) & (y_pred_mc == 2)
idx_bad = np.where(bad)[0][:15]
demo = X_test.iloc[idx_bad].copy()
demo["actual_class"] = y_mc_test[idx_bad]
demo["pred_class"] = y_pred_mc[idx_bad]
demo["price"] = df.loc[X_test.index[idx_bad], "price"].values
demo["brand"] = df.loc[X_test.index[idx_bad], "brand"].values
print("Sample Extreme Tax -> Brand Premium confusions:")
print(demo.head(10).to_string())

## Part 5 — Multinomial Logistic odds ratios (interpretability)

In [ ]:
lr = lr_best.named_steps["clf"]
coef = lr.coef_
odds = np.exp(coef)
class_names = ["Steal", "Fair", "Brand Premium", "Extreme Tax"]
for k, cname in enumerate(class_names):
    s = pd.Series(odds[k], index=feature_names).sort_values(ascending=False)
    print(f"\n=== {cname} — top / bottom odds ratios ===")
    print(pd.concat([s.head(8), s.tail(8)]).round(3).to_string())

## Part 6 — Decision tree (top levels)

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(dt_best, max_depth=3, feature_names=list(feature_names), class_names=CLASS_NAMES, filled=True, ax=ax, fontsize=8)
plt.title("Decision tree (tuned) — depth 3 of full tree")
plt.tight_layout()
plt.show()

## Part 7 — Binary robustness: Steal vs not

In [ ]:
bin_models = {
    "Logistic": clone(lr_best),
    "HistGBM": clone(hgb_best),
}
for name, m in bin_models.items():
    m.fit(Xtr_enc, y_bin_train)
    p = m.predict_proba(Xte_enc)[:, 1]
    yhat = (p >= 0.5).astype(int)
    print(name, "ROC-AUC", roc_auc_score(y_bin_test, p).round(4), "AP", average_precision_score(y_bin_test, p).round(4))

## Part 8 — Persist artifact

In [ ]:
artifact = {
    "preprocessor": preprocessor,
    "model": winner,
    "feature_names": feature_names,
    "value_score_quantiles_train_usd": {"q10": float(q10), "q50": float(q50), "q90": float(q90)},
    "random_search": {
        "cv_splits": 3,
        "tuning_rows": int(TUNE_SUBSAMPLE),
        "n_iter": N_ITER_SEARCH,
        "seed": SEED,
        "note": "Hyperparameters chosen on stratified 25k subset; models refit on full 80k train.",
    },
}
joblib.dump(artifact, MODEL_PATH)
print("Saved:", MODEL_PATH)

## Part 9 — Next steps

Final report scaffold: [reports/outline.md](../../reports/outline.md).

Re-run `Project_refactored.ipynb` if residuals change; then re-run this notebook.